# Pretrained models for spatial proteomics
| Model | Use in this project | Weights |
|---|---|---|
| **KRONOS** (Mahmood lab) | marker-aware patch/cell embeddings → phenotyping, patient stratification (MIL), retrieval | HF `MahmoodLab/kronos` (**gated**: request access, then `huggingface-cli login`) |
| **VirTues** (Bunne lab) | cell/niche/tissue tokens; virtual staining of missing markers; "virtual biomarker discovery" | HF `bunnelab/virtues` (`virtues-sp31` MIT, `virtues-sp32` CC BY-NC) |
| **Mesmer** (DeepCell) | pretrained whole-cell segmentation (nuclear + membrane channel) | auto-download (needs `DEEPCELL_ACCESS_TOKEN`) |
| **Cellpose** | pretrained generalist segmentation | auto-download |

> Template. GPU recommended. Never commit tokens. Set them as environment variables.

## 1. KRONOS
```bash
git clone https://github.com/mahmoodlab/KRONOS.git && cd KRONOS && pip install -e .[tutorials]
huggingface-cli login     # after access to MahmoodLab/kronos is granted
```

In [ ]:
import torch
from kronos import create_model_from_pretrained

model, precision, embedding_dim = create_model_from_pretrained(
    checkpoint_path="hf_hub:MahmoodLab/kronos", cache_dir="./model_assets")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device).eval()
print(precision, embedding_dim)

In [ ]:
# Input: (batch, n_markers, 224, 224) tiles cut from a multiplexed image, each marker channel normalised
# with the per-marker mean/std from KRONOS's marker_metadata.csv (match your panel names to its marker ids).
import numpy as np, pandas as pd
meta = pd.read_csv("KRONOS/marker_metadata.csv")          # adjust path to your clone
print(meta.head())

def tiles_around_cells(img, centroids, size=224):
    """img: (C, H, W) array; centroids: (n, 2) array of (x, y). Returns (n, C, size, size) tiles."""
    h = size // 2
    pad = np.pad(img, ((0, 0), (h, h), (h, h)), mode="constant")
    return np.stack([pad[:, int(y):int(y) + size, int(x):int(x) + size] for x, y in centroids])

# batch = torch.tensor(tiles_around_cells(img, xy), dtype=torch.float32, device=device)
batch = torch.randn(2, 10, 224, 224, device=device)        # dummy
mean, std = torch.zeros(10, device=device), torch.ones(10, device=device)   # replace with metadata values
batch = (batch - mean[None, :, None, None]) / std[None, :, None, None]
with torch.no_grad():
    patch_emb, marker_emb, token_emb = model(batch)
patch_emb.shape, marker_emb.shape, token_emb.shape

**Use in this project:** frozen KRONOS / VirTues embeddings + a linear probe are protein-only baselines for H1 (cell typing, niches) and for the H2 student. They never see RNA, so they bound what protein alone achieves.

## 2. VirTues
```bash
git clone https://github.com/bunnelab/virtues.git && cd virtues
source setup.sh && conda activate virtues && pip install -e .
```

In [ ]:
from huggingface_hub import hf_hub_download
ckpt = hf_hub_download(repo_id="bunnelab/virtues", filename="virtues-sp31/model.safetensors",
                       cache_dir="./model_assets")
print(ckpt)
# Loading code and data format: see virtues/notebooks/
#   1_demo_reconstruction.ipynb   - inpaint masked markers (virtual staining)
#   2_demo_cell_phenotyping.ipynb - cell tokens → phenotyping & virtual biomarker discovery
#   3_demo_segmentation.ipynb     - joint cell + cell-type segmentation
# Marker embeddings come from ESM-2 protein language model:
#   python -m virtues.utils.download_fastas --output_dir fastas --input panel.csv --id_column uniprot
#   python -m virtues.utils.compute_esm_embeddings --input_dir fastas --output_dir esm --device cuda
# Data: convert to spora [data] format (https://spora.epfl.ch/datasets.html, loader: bunnelab/spora-io)

## 3. Upstream segmentation: Mesmer and Cellpose

In [ ]:
# Mesmer (pip install deepcell); needs env var DEEPCELL_ACCESS_TOKEN from https://users.deepcell.org
from deepcell.applications import Mesmer
app = Mesmer()
# im: (batch, H, W, 2) with channel 0 = nuclear (e.g. DNA/Histone H3), channel 1 = membrane (e.g. CD45+panCK sum)
# masks = app.predict(im, image_mpp=0.5, compartment="whole-cell")

# Cellpose (pip install cellpose)
from cellpose import models
cp = models.CellposeModel(gpu=torch.cuda.is_available(), model_type="cyto3")
# masks, flows, styles = cp.eval(img_2ch, diameter=None, channels=[2, 1])
# Then quantify mean marker intensity per mask (e.g. skimage.measure.regionprops_table) → cell x marker matrix.